### Import required packages

In [1]:
import pandas as pd
import xarray as xr
import s3fs
import pytest
import os
import sys
import numpy as np
import apache_beam as beam
import xarray_beam as xbeam
import datetime

### Reorganize paths

In [2]:
module_dir = os.path.abspath('../src/main/')
sys.path.insert(0, module_dir)

### Load test datasets

In [ ]:
retrospective_data_2849991 = pd.read_csv(
    "../tests/test_data/retrospective_output_data_2849991.csv",
    index_col=0)
retrospective_data_6010106 = pd.read_csv(
    "../tests/test_data/retrospective_output_data_6010106.csv",
    index_col=0)
retrospective_data_942030011 = pd.read_csv(
    "../tests/test_data/retrospective_output_data_942030011.csv",
    index_col=0)

daily_retro_data_raw_2849991 = pd.read_csv(
    "../tests/test_data/streamflow_df_for_indices_computation_2849991.csv",
    index_col=0)
daily_retro_data_raw_6010106 = pd.read_csv(
    "../tests/test_data/streamflow_df_for_indices_computation_6010106.csv",
    index_col=0)
daily_retro_data_raw_942030011 = pd.read_csv(
    "../tests/test_data/streamflow_df_for_indices_computation_942030011.csv",
    index_col=0)

daily_retro_data_2849991 = pd.read_csv(
    "../tests/test_data/streamflow_df_with_water_year_for_indices_computation_2849991.csv",
    index_col=0)
daily_retro_data_6010106 = pd.read_csv(
    "../tests/test_data/streamflow_df_with_water_year_for_indices_computation_6010106.csv", 
    index_col=0)
daily_retro_data_942030011 = pd.read_csv(
    "../tests/test_data/streamflow_df_with_water_year_for_indices_computation_942030011.csv", 
    index_col=0)
daily_retro_data_intermittent = pd.read_csv(
    "../tests/test_data/streamflow_df_with_water_year_for_indices_computation_synthetic.csv", 
    index_col=0)

### Test santizing operation before writing

In [ ]:
from dataflow_nwm_retro_indices_transformation import SanitizeNaNsDoFn

sample_retro_records = [{'feature_id': 2849991, 'time': pd.to_datetime('1979-02-01 01:00:00'), 'streamflow': np.nan, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 210.77301025390625, 'qBucket': 0.004809999838471413, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0},
{'feature_id': 2849991, 'time': pd.to_datetime('1979-02-01 02:00:00'), 'streamflow': 590.9599609375, 'velocity': np.nan, 'qBtmVertRunoff': 206.4910125732422, 'qBucket': 0.004819999914616346, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0},
{'feature_id': 2849991, 'time': pd.to_datetime('1979-02-01 03:00:00'), 'streamflow': 590.4400024414062, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': np.nan, 'qBucket': 0.00482999999076128, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0},
{'feature_id': 2849991, 'time': pd.to_datetime('1979-02-01 04:00:00'), 'streamflow': 589.9299926757812, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 206.25601196289062, 'qBucket': np.nan, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0},
{'feature_id': 2849991, 'time': pd.to_datetime('1979-02-01 05:00:00'), 'streamflow': 589.4299926757812, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 215.49801635742188, 'qBucket': 0.004859999753534794, 'qSfcLatRunoff': np.nan, 'q_lateral': 0.0}]

with beam.Pipeline() as pipeline:
    input_collection = pipeline | "Create Sample Records" >> beam.Create(sample_retro_records)
    sanitized_collection = input_collection | "Sanitize NaNs" >> beam.ParDo(SanitizeNaNsDoFn())
    sanitized_collection | "Print Sanitized Records" >> beam.Map(print)

2026-02-21 09:50:39,064 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
2026-02-21 09:50:39,125 - INFO - Creating state cache with size 104857600


{'feature_id': 2849991, 'time': Timestamp('1979-02-01 01:00:00'), 'streamflow': nan, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 210.77301025390625, 'qBucket': 0.004809999838471413, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 02:00:00'), 'streamflow': 590.9599609375, 'velocity': nan, 'qBtmVertRunoff': 206.4910125732422, 'qBucket': 0.004819999914616346, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 03:00:00'), 'streamflow': 590.4400024414062, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': nan, 'qBucket': 0.00482999999076128, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 04:00:00'), 'streamflow': 589.9299926757812, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 206.25601196289062, 'qBucket': nan, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 05:00:00'), 'streamflow': 589.4299926757812,

In [11]:
sample_indices_records = [{'reach_id': 101, 'monthwise_mean': [10, 11, 12], 'monthwise_cov': [1, np.nan, 3], 'nth_percentile_flows': [5, 10, 15], 'variability_index': np.nan, 'slope_fdc': 0.4, 'flashiness_index': 0.3, 'sevenQ10': 0.2, 'baseflow_index': np.nan,'mean_annual_7_day_min': 0.15,'zero_flow_days_n': 7,'low_flow_days_n': 10,'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363}, 
                          {'reach_id': 107, 'monthwise_mean': [10, np.nan, 12], 'monthwise_cov': [1, 2, 3], 'nth_percentile_flows': [5, 10, 15], 'variability_index': 0.5, 'slope_fdc': np.nan, 'flashiness_index': 0.3, 'sevenQ10': 0.2, 'baseflow_index': 0.1,'mean_annual_7_day_min': np.nan,'zero_flow_days_n': 7,'low_flow_days_n': 10,'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363},
                          {'reach_id': 109, 'monthwise_mean': [10, 11, 12], 'monthwise_cov': [1, 2, 3], 'nth_percentile_flows': [5, 10, 15], 'variability_index': 0.5, 'slope_fdc': 0.4, 'flashiness_index': 0.3, 'sevenQ10': 0.2, 'baseflow_index': 0.1,'mean_annual_7_day_min': 0.15,'zero_flow_days_n': 7,'low_flow_days_n': 10,'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363},
                          {'reach_id': 110, 'monthwise_mean': [np.nan, np.nan, np.nan], 'monthwise_cov': [np.nan, np.nan, np.nan], 'nth_percentile_flows': [5, 10, 15], 'variability_index': 0.5, 'slope_fdc': 0.4, 'flashiness_index': np.nan, 'sevenQ10': 0.2, 'baseflow_index': 0.1,'mean_annual_7_day_min': 0.15,'zero_flow_days_n': np.nan,'low_flow_days_n': 10,'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363},
                          {'reach_id': 120, 'monthwise_mean': [np.nan, np.nan, np.nan], 'monthwise_cov': [1, 2, 3], 'nth_percentile_flows': [5, 10, 15], 'variability_index': 0.5, 'slope_fdc': 0.4, 'flashiness_index': 0.3, 'sevenQ10': np.nan, 'baseflow_index': 0.1,'mean_annual_7_day_min': 0.15,'zero_flow_days_n': 7,'low_flow_days_n': 10,'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363},
                          {'reach_id': 111, 'monthwise_mean': [np.nan, np.nan, np.nan], 'monthwise_cov': [np.nan, np.nan, np.nan], 'nth_percentile_flows': [np.nan, np.nan, np.nan], 'variability_index': np.nan, 'slope_fdc': np.nan, 'flashiness_index': np.nan, 'sevenQ10': np.nan, 'baseflow_index': np.nan,'mean_annual_7_day_min': np.nan,'zero_flow_days_n': np.nan,'low_flow_days_n': np.nan,'duration_low_flow_event': np.nan, 'high_flow_days_n': np.nan, 'duration_high_flow_event': np.nan, 'half_flow_date': np.nan, 'starting_date_of_flood_season': np.nan}]

with beam.Pipeline() as pipeline:
    input_collection = pipeline | "Create Sample Indices Records" >> beam.Create(sample_indices_records)
    sanitized_collection = input_collection | "Sanitize NaNs in Indices Records" >> beam.ParDo(SanitizeNaNsDoFn())
    sanitized_collection | "Print Sanitized Indices Records" >> beam.Map(print)

2026-02-21 09:50:41,965 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
2026-02-21 09:50:42,024 - INFO - Creating state cache with size 104857600


{'reach_id': 101, 'monthwise_mean': [10, 11, 12], 'monthwise_cov': [1, nan, 3], 'nth_percentile_flows': [5, 10, 15], 'variability_index': nan, 'slope_fdc': 0.4, 'flashiness_index': 0.3, 'sevenQ10': 0.2, 'baseflow_index': nan, 'mean_annual_7_day_min': 0.15, 'zero_flow_days_n': 7, 'low_flow_days_n': 10, 'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363}
{'reach_id': 107, 'monthwise_mean': [10, nan, 12], 'monthwise_cov': [1, 2, 3], 'nth_percentile_flows': [5, 10, 15], 'variability_index': 0.5, 'slope_fdc': nan, 'flashiness_index': 0.3, 'sevenQ10': 0.2, 'baseflow_index': 0.1, 'mean_annual_7_day_min': nan, 'zero_flow_days_n': 7, 'low_flow_days_n': 10, 'duration_low_flow_event': 15, 'high_flow_days_n': 11, 'duration_high_flow_event': 13, 'half_flow_date': 263, 'starting_date_of_flood_season': 363}
{'reach_id': 109, 'monthwise_mean': [10, 11, 12], 'monthwise_cov': [1, 2, 3], 'nth_percentile_flows'

### Testing the conversion from calendar dates to water years

In [ ]:
from dataflow_nwm_retro_indices_transformation import get_water_years

def test_water_years():
    test_dates = pd.Series(['1979-01-01', '1979-10-01', '1980-02-15', '1980-09-30', '1980-10-03'])
    expected_water_years = [1979, 1980, 1980, 1980, 1981]
    computed_water_years = get_water_years(test_dates).to_list()
    for date, computed, expected in zip(test_dates, computed_water_years, expected_water_years):
        print(f"Date {date}, Computed: {computed}, Expected: {expected}")
    assert computed_water_years == expected_water_years, f"Expected {expected_water_years} but got {computed_water_years}"
    print(" -> Test passed (OK)")

test_water_years()

Date 1979-01-01, Computed: 1979, Expected: 1979
Date 1979-10-01, Computed: 1980, Expected: 1980
Date 1980-02-15, Computed: 1980, Expected: 1980
Date 1980-09-30, Computed: 1980, Expected: 1980
Date 1980-10-03, Computed: 1981, Expected: 1981
 -> Test passed (OK)


### Test converting to water year data

In [ ]:
from dataflow_nwm_retro_indices_transformation import convert_to_water_year_data

def test_convert_to_water_year_data():
    test_data = pd.DataFrame({
        'date': ['2020-01-01', '2020-10-01', '2021-02-15', '2021-09-30', '2021-10-03', '2022-01-01', '2022-09-29', '2023-10-01'],
        'streamflow': [1, 2, 3, 4, 5, 6, 7, 8]
    })
    expected_output = pd.DataFrame({
        'date': ['2020-10-01', '2021-02-15', '2021-09-30', '2021-10-03', '2022-01-01', '2022-09-29'],
        'streamflow': [2, 3, 4, 5, 6, 7],
        'water_year': [2021, 2021, 2021, 2022, 2022, 2022],
    })
    expected_output['water_year'] = expected_output['water_year'].astype('int32')
    output = convert_to_water_year_data(test_data).reset_index(drop=True)
    print(output)
    pd.testing.assert_frame_equal(output, expected_output)
    print(" -> Test passed (OK)")

test_convert_to_water_year_data()

         date  streamflow  water_year
0  2020-10-01           2        2021
1  2021-02-15           3        2021
2  2021-09-30           4        2021
3  2021-10-03           5        2022
4  2022-01-01           6        2022
5  2022-09-29           7        2022
 -> Test passed (OK)


In [16]:
def test_convert_to_water_year_data_actual(data,  expected_data):
    converted_data = convert_to_water_year_data(data)
    assert not converted_data.empty, "Converted data is empty"
    assert 'water_year' in converted_data.columns, "Water year column is missing"
    assert np.allclose(converted_data['water_year'], expected_data['water_year']), "Water year data does not match"
    pd.testing.assert_frame_equal(converted_data, expected_data, check_dtype=False)
    print(" -> Test passed (OK)")
test_convert_to_water_year_data_actual(daily_retro_data_raw_2849991, daily_retro_data_2849991)
test_convert_to_water_year_data_actual(daily_retro_data_raw_6010106, daily_retro_data_6010106)
test_convert_to_water_year_data_actual(daily_retro_data_raw_942030011, daily_retro_data_942030011)

 -> Test passed (OK)
 -> Test passed (OK)
 -> Test passed (OK)


### Test constructing (feature_id, date) based key

In [ ]:
from dataflow_nwm_retro_indices_transformation import extract_date_key

def test_extract_date_key():
    test_data = [{'feature_id': 101, "time": "2000-01-01 00:00:00", "streamflow": 100.0},
                    {'feature_id': 101, "time": "2000-01-01 01:00:00", "streamflow": 110.0},
                    {'feature_id': 102, "time": "2000-02-28 23:00:00", "streamflow": 120.0}]
    expected_outputs = [((101, '2000-01-01'), 100.0), ((101, '2000-01-01'), 110.0), ((102, '2000-02-28'), 120.0)]
    
    assert len(test_data) == len(expected_outputs), "Test data and expected outputs must have the same length"
    for record, expected in zip(test_data, expected_outputs):
        output = extract_date_key(record)
        print(f"Input: {record} -> Output: {output}")
        assert output == expected, f"Expected {expected} but got {output}"
    print(" -> Test passed (OK)")
    
test_extract_date_key()

Input: {'feature_id': 101, 'time': '2000-01-01 00:00:00', 'streamflow': 100.0} -> Output: ((101, '2000-01-01'), 100.0)
Input: {'feature_id': 101, 'time': '2000-01-01 01:00:00', 'streamflow': 110.0} -> Output: ((101, '2000-01-01'), 110.0)
Input: {'feature_id': 102, 'time': '2000-02-28 23:00:00', 'streamflow': 120.0} -> Output: ((102, '2000-02-28'), 120.0)
 -> Test passed (OK)


### Testing the monthwise mean and cov calculation

In [ ]:
from dataflow_nwm_retro_indices_transformation import monthwise_mean_and_cov
def test_monthwise_stats_for_synthetic_data(streamflow_list, month_means, month_cvs):
    dates = pd.date_range(start="2000-01-01", periods=36, freq="MS")
    streamflow = streamflow_list
    synthetic_ts = pd.DataFrame({
        "date": dates,
        "streamflow": streamflow
    }).set_index("date")
    monthwise_means, monthwise_covs = monthwise_mean_and_cov(synthetic_ts)
    print(f"Computed monthwise means: {monthwise_means}")
    print(f"Computed monthwise CVs: {monthwise_covs}")
    assert np.allclose(monthwise_means, month_means, atol=0.1), f"Expected means {month_means} but got {monthwise_means}"
    assert np.allclose(monthwise_covs, month_cvs, atol=0.1), f"Expected CVs {month_cvs} but got {monthwise_covs}"
    print(" -> Test passed (OK)")
    

# Case 1: Streamflow with same monthwise means and CVs
streamflow_series_1 = [10,] * 12 + [15,] * 12 + [20,] * 12
means_1 = [15,] * 12
cvs_1 = [33.33333333,] * 12
test_monthwise_stats_for_synthetic_data(streamflow_series_1, means_1, cvs_1)

# Case 2: Streamflow with varying monthwise means, but same SD (and thus varying CVs)
streamflow_series_2 = [1,2,3,4,5,6,7,8,9,10,11,12] + [13,14,15,16,17,18,19,20,21,22,23,24] + [25,26,27,28,29,30,31,32,33,34,35,36]
means_2 = [13,14,15,16,17,18,19,20,21,22,23,24]
sd_2 = [12,] * 12
cvs_2 = [sd * 100 / mean for sd, mean in zip(sd_2, means_2)]
test_monthwise_stats_for_synthetic_data(streamflow_series_2, means_2, cvs_2)

Computed monthwise means: [15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0]
Computed monthwise CVs: [33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33]
 -> Test passed (OK)
Computed monthwise means: [13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0]
Computed monthwise CVs: [92.31, 85.71, 80.0, 75.0, 70.59, 66.67, 63.16, 60.0, 57.14, 54.55, 52.17, 50.0]
 -> Test passed (OK)


### Testing event duration calculation

In [ ]:
from dataflow_nwm_retro_indices_transformation import get_event_durations

def test_get_event_durations(event_series, expected_durations):
    computed_durations = get_event_durations(event_series)
    assert computed_durations == expected_durations, f"Expected {expected_durations}, but got {computed_durations}"
    print(f"Input event series: {event_series} -> Computed durations: {computed_durations} -> Test passed (OK)")
    print(" -> Test passed (OK)")


# Case 1
event_series_1 = [0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0]
expected_durations_1 = [2, 3]
test_get_event_durations(event_series_1, expected_durations_1)

# Case 2
event_series_2 = [1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1]
expected_durations_2 = [2, 1, 2, 3]
test_get_event_durations(event_series_2, expected_durations_2)

Input event series: [0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0] -> Computed durations: [2, 3] -> Test passed (OK)
 -> Test passed (OK)
Input event series: [1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1] -> Computed durations: [2, 1, 2, 3] -> Test passed (OK)
 -> Test passed (OK)


## Check computations of different indices

In [ ]:

daily_retro_data_2849991 = daily_retro_data_2849991.set_index('date')
daily_retro_data_2849991.index = pd.to_datetime(daily_retro_data_2849991.index)
daily_retro_data_2849991 = daily_retro_data_2849991.sort_values(['water_year', daily_retro_data_2849991.index.name])


daily_retro_data_6010106 = daily_retro_data_6010106.set_index('date')
daily_retro_data_6010106.index = pd.to_datetime(daily_retro_data_6010106.index)
daily_retro_data_6010106 = daily_retro_data_6010106.sort_values(['water_year', daily_retro_data_6010106.index.name])


daily_retro_data_942030011 = daily_retro_data_942030011.set_index('date')
daily_retro_data_942030011.index = pd.to_datetime(daily_retro_data_942030011.index)
daily_retro_data_942030011 = daily_retro_data_942030011.sort_values(['water_year', daily_retro_data_942030011.index.name])


daily_retro_data_intermittent = daily_retro_data_intermittent.set_index('date')
daily_retro_data_intermittent.index = pd.to_datetime(daily_retro_data_intermittent.index)
daily_retro_data_intermittent = daily_retro_data_intermittent.sort_values(['water_year', daily_retro_data_intermittent.index.name])

daily_retro_data_2849991.head()
 

,streamflow,water_year
date,,
1979-10-01,149.423331,1980
1979-10-02,148.274997,1980
1979-10-03,147.153329,1980
1979-10-04,146.091663,1980
1979-10-05,145.044997,1980


### Test computing monthwise mean and cov

In [ ]:
from dataflow_nwm_retro_indices_transformation import monthwise_mean_and_cov

def test_monthwise_mean_and_cov_on_retro_data(streamflow_df, expected_means, expected_covs):
    monthwise_means, monthwise_covs = monthwise_mean_and_cov(streamflow_df)
    print(f"Computed monthwise means: \n{monthwise_means}", f"\nExpected monthwise means: \n{expected_means}")
    print(f"Computed monthwise covs: \n{monthwise_covs}", f"\nExpected monthwise covs: \n{expected_covs}")
    for i in range(len(expected_means)):
        assert np.allclose(monthwise_means[i], expected_means[i], atol=0.001*expected_means[i]), f"Expected means {expected_means[i]} but got {monthwise_means[i]}"
        assert np.allclose(monthwise_covs[i], expected_covs[i], atol=0.001*expected_covs[i]), f"Expected covs {expected_covs[i]} but got {monthwise_covs[i]}"
    print(" -> Test passed (OK)")

test_monthwise_mean_and_cov_on_retro_data(daily_retro_data_2849991, 
                                          expected_means=[571.9318, 731.2458, 793.3216, 616.4847, 468.0034, 355.1383, 260.5979, 196.5698, 153.3890, 126.6037, 134.2540, 303.9444], 
                                          expected_covs=[115.5004394, 100.6381749, 83.91054507, 59.93274843, 49.8302102, 45.73212789, 39.25690667, 34.14860613, 29.55423103, 25.78400451, 84.8216531, 121.7916964])
test_monthwise_mean_and_cov_on_retro_data(daily_retro_data_6010106, 
                                          expected_means=[2535.231, 2533.408, 2894.204, 3268.356, 4040.579, 4295.923, 3710.290, 2897.151, 2702.426, 2699.667, 2517.985, 2587.835], 
                                          expected_covs=[43.62607905, 39.10017762, 48.17040547, 48.22224015, 52.77632817, 54.64111741, 56.28839742, 47.31726564, 51.07039626, 62.72674313, 42.4175384, 46.86886775])
test_monthwise_mean_and_cov_on_retro_data(daily_retro_data_942030011, 
                                          expected_means=[81.88639, 89.07923, 114.26536, 104.84186, 160.18195, 162.50388, 92.82083, 66.06341, 74.20742, 104.44780, 97.97152, 95.87465], 
                                          expected_covs=[109.0227085,104.0279999,108.5542668,92.40490013,139.9024016,115.6196809,114.502888,103.5482576,144.346089,170.1864739, 135.2582505, 151.1353114])



Computed monthwise means: 
[571.93, 731.25, 793.32, 616.48, 468.0, 355.14, 260.6, 196.57, 153.39, 126.6, 134.25, 303.94] 
Expected monthwise means: 
[571.9318, 731.2458, 793.3216, 616.4847, 468.0034, 355.1383, 260.5979, 196.5698, 153.389, 126.6037, 134.254, 303.9444]
Computed monthwise covs: 
[115.5, 100.64, 83.91, 59.93, 49.83, 45.73, 39.26, 34.15, 29.55, 25.78, 84.82, 121.79] 
Expected monthwise covs: 
[115.5004394, 100.6381749, 83.91054507, 59.93274843, 49.8302102, 45.73212789, 39.25690667, 34.14860613, 29.55423103, 25.78400451, 84.8216531, 121.7916964]
 -> Test passed (OK)
Computed monthwise means: 
[2535.23, 2533.41, 2894.2, 3268.36, 4040.58, 4295.92, 3710.29, 2897.15, 2702.43, 2699.67, 2517.99, 2587.83] 
Expected monthwise means: 
[2535.231, 2533.408, 2894.204, 3268.356, 4040.579, 4295.923, 3710.29, 2897.151, 2702.426, 2699.667, 2517.985, 2587.835]
Computed monthwise covs: 
[43.63, 39.1, 48.17, 48.22, 52.78, 54.64, 56.29, 47.32, 51.07, 62.73, 42.42, 46.87] 
Expected monthwise cov

### Test computing percentiles

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_percentiles

def test_compute_percentiles_on_retro_data(streamflow_df, expected_percentiles):
    percentiles_to_compute = [2,5,10,15,20,25,30,33,35,40,45,50,55,60,65,66,70,75,80,85,90,95,99]
    computed_percentiles = compute_percentiles(streamflow_df['streamflow'], percentiles_to_compute)
    for p in range(len(percentiles_to_compute)):
        assert np.isclose(computed_percentiles[p], expected_percentiles[p], atol=0.001*expected_percentiles[p]), f"Expected {percentiles_to_compute[p]}th percentile to be {expected_percentiles[p]} but got {computed_percentiles[p]}"
    print(f"Computed percentiles: \n{computed_percentiles}", f"\nExpected percentiles: \n{expected_percentiles}")
    print(" -> Test passed (OK)")
    return computed_percentiles
    
    

nth_percentiles_2849991 = test_compute_percentiles_on_retro_data(daily_retro_data_2849991, 
                                       expected_percentiles=[77.805615, 86.229915, 101.341123, 114.366164, 126.679247, 139.297497, 153.24833, 163.221793, 169.945186, 189.985329, 212.066077, 237.807702, 268.187515, 305.600326, 351.134429, 360.720058, 405.414492, 466.691344, 548.368652, 659.070716, 833.441855, 1181.086948, 2337.361222])
nth_percentiles_6010106 = test_compute_percentiles_on_retro_data(daily_retro_data_6010106,
                                       expected_percentiles=[1420.203545, 1518.181025, 1662.699378, 1785.781073, 1895.940955, 2015.481525, 2107.00165, 2162.731471, 2203.450928, 2311.34739, 2427.846697, 2564.0012, 2694.904692, 2844.533872, 3044.130571, 3090.168263, 3269.372549, 3536.534828, 3896.954419, 4378.778582, 5055.943919, 6121.454435, 9612.994579])
nth_percentiles_942030011 = test_compute_percentiles_on_retro_data(daily_retro_data_942030011,
                                       expected_percentiles=[24.524724, 27.468208, 31.224458, 34.62327, 37.897332, 40.755103, 43.579999, 45.576878, 46.869041, 50.723999, 55.140478, 59.71479, 65.486436, 72.841998, 81.626331, 83.91939, 93.645789, 108.060935, 129.020581, 158.931539, 210.264579, 313.578099, 729.038949])
nth_percentiles_intermittent = test_compute_percentiles_on_retro_data(daily_retro_data_intermittent,
                                       expected_percentiles=[77.51184,  86.015061, 101.134623, 114.241789, 126.537747, 139.233643, 153.091123, 163.150771, 169.752976, 189.820579, 211.785891, 237.557909, 268.017721, 305.10016, 350.66301, 360.115757, 404.902533, 466.416136, 548.170738, 658.79686, 833.375646, 1181.086948, 2337.361222])

Computed percentiles: 
[77.80561479568482, 86.22991528511048, 101.34112278620402, 114.36616430282594, 126.6792474746704, 139.29749727249146, 153.24833011627197, 163.22179332733154, 169.9451858838399, 189.98532880147297, 212.06607729593912, 237.8077023824056, 268.1875151316325, 305.6003262837728, 351.13442942301424, 360.72005813598633, 405.41449152628576, 466.69134362538654, 548.36865234375, 659.0707155863446, 833.4418553670247, 1181.086947886149, 2337.3612219238285] 
Expected percentiles: 
[77.805615, 86.229915, 101.341123, 114.366164, 126.679247, 139.297497, 153.24833, 163.221793, 169.945186, 189.985329, 212.066077, 237.807702, 268.187515, 305.600326, 351.134429, 360.720058, 405.414492, 466.691344, 548.368652, 659.070716, 833.441855, 1181.086948, 2337.361222]
 -> Test passed (OK)
Computed percentiles: 
[1420.2035450236003, 1518.1810251871746, 1662.6993779500326, 1785.781072998047, 1895.9409545898436, 2015.4815254211426, 2107.001649983724, 2162.7314709472657, 2203.450927734375, 2311.34

### Test computing 7Q10 and MAM7

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_7Q10_and_MAM7

def test_compute_7Q10_and_MAM7(data, expected_7Q10, expected_mam7):
    seven_q_10, mam7 = compute_7Q10_and_MAM7(data)
    print(f"Computed 7Q10: {seven_q_10}, Expected 7Q10: {expected_7Q10}")
    print(f"Computed MAM7: {mam7}, Expected MAM7: {expected_mam7}")
    assert np.isclose(seven_q_10, expected_7Q10, atol=0.01*expected_7Q10), f"Expected 7Q10 {expected_7Q10} but got {seven_q_10}"
    assert np.isclose(mam7, expected_mam7, atol=0.01*expected_mam7), f"Expected MAM7 {expected_mam7} but got {mam7}"
    print(" -> Test passed (OK)")

test_compute_7Q10_and_MAM7(data = daily_retro_data_2849991, expected_7Q10 = 70.91965, expected_mam7 = 97.58392)
test_compute_7Q10_and_MAM7(data = daily_retro_data_6010106, expected_7Q10 = 1430.36, expected_mam7 = 1882.756)
test_compute_7Q10_and_MAM7(data = daily_retro_data_942030011, expected_7Q10 = 22.8686, expected_mam7 = 30.44139)
test_compute_7Q10_and_MAM7(data = daily_retro_data_intermittent, expected_7Q10 = 65.95, expected_mam7 = 92.11101) # 2 years with zero flows

Computed 7Q10: 71.01974424687327, Expected 7Q10: 70.91965
Computed MAM7: 97.93485242636629, Expected MAM7: 97.58392
 -> Test passed (OK)
Computed 7Q10: 1426.2203535135245, Expected 7Q10: 1430.36
Computed MAM7: 1885.4279704046405, Expected MAM7: 1882.756
 -> Test passed (OK)
Computed 7Q10: 22.8294757299419, Expected 7Q10: 22.8686
Computed MAM7: 30.486148264981008, Expected MAM7: 30.44139
 -> Test passed (OK)
Computed 7Q10: 65.9528485231293, Expected 7Q10: 65.95
Computed MAM7: 92.46194557537005, Expected MAM7: 92.11101
 -> Test passed (OK)


### Test computing baseflow index

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_bfi
from baseflow.comparision import strict_baseflow
from baseflow.param_estimate import recession_coefficient, param_calibrate
from baseflow.methods import Eckhardt, LH

def bfi_step_by_step(streamflow_df):
    streamflow_series = streamflow_df['streamflow'].values
    strict_series = strict_baseflow(streamflow_series)
    lh_series = LH(streamflow_series)
    alpha = recession_coefficient(streamflow_series, strict_series)
    print("recession_alpha:", alpha)
    bfi_max = param_calibrate(np.arange(0.001, 1, 0.001), Eckhardt, streamflow_series, lh_series, alpha)
    print("maximum_bfi:", bfi_max)
    baseflow_series = Eckhardt(streamflow_series, lh_series, alpha, bfi_max)
    lh_bfi = np.mean(lh_series) / np.mean(streamflow_series)
    print("LH BFI:", lh_bfi)
    bfi = np.mean(baseflow_series) / np.mean(streamflow_series)
    return bfi

def test_compute_bfi(data):
    pipe_bfi = compute_bfi(data)
    local_bfi = bfi_step_by_step(data)
    print(f"BFI from compute_bfi function: {pipe_bfi}, \nBFI from step-by-step computation: {local_bfi}")
    assert np.isclose(pipe_bfi, local_bfi, atol=0.01), f"Expected BFI {local_bfi} but got {pipe_bfi}"
    print(" -> Test passed (OK)")
    
test_compute_bfi(data = daily_retro_data_2849991)
test_compute_bfi(data = daily_retro_data_6010106)
test_compute_bfi(data = daily_retro_data_942030011)

100%|██████████| 1/1 [00:00<00:00, 18.06it/s]


recession_alpha: 0.9952132479992184
maximum_bfi: 0.47800000000000004
LH BFI: 0.7871956053836542
BFI from compute_bfi function: 0.4528499908640328, 
BFI from step-by-step computation: 0.4528499908640328
 -> Test passed (OK)


100%|██████████| 1/1 [00:00<00:00, 19.22it/s]


recession_alpha: 0.997881866505049
maximum_bfi: 0.929
LH BFI: 0.8664327095645044
BFI from compute_bfi function: 0.8519715282134741, 
BFI from step-by-step computation: 0.8519715282134741
 -> Test passed (OK)


100%|██████████| 1/1 [00:00<00:00, 18.52it/s]


recession_alpha: 0.9934591352836328
maximum_bfi: 0.63
LH BFI: 0.63998018728772
BFI from compute_bfi function: 0.5377454693897243, 
BFI from step-by-step computation: 0.5377454693897243
 -> Test passed (OK)


### Test computing variability index

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_variability_index

def test_compute_variability_index(nth_percentiles, n, expected):
    variability_index = compute_variability_index(nth_percentiles, n)
    print(f"Computed Variability Index: {variability_index}, Expected: {expected}")
    assert np.isclose(variability_index, expected, atol=0.01*expected), f"Expected Variability Index {expected} but got {variability_index}"
    print(" -> Test passed (OK)")
    
def vi_alt_comp(all_percentiles):
    # [2,5,10,15,20,25,30,33,35,40,45,50,55,60,65,66,70,75,80,85,90,95,99]
    vi_percentiles = [all_percentiles[i] for i in [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19 , 20, 21]]
    vi_percentiles = [vi_percentiles[i] if vi_percentiles[i] > 0.001 else 0.001 for i in range(len(vi_percentiles))]
    log_D = np.log10(vi_percentiles)
    log_D_mean = np.mean(log_D)
    vi = np.sqrt(np.sum((log_D - log_D_mean)**2) / (18))
    return vi

synthetic_nth_percentile_flows = nth_percentiles_intermittent.copy()
# test for VI percentile zero flows
synthetic_nth_percentile_flows[0:3] = [0,0,0]

expected_vi_2849991 = vi_alt_comp(nth_percentiles_2849991)
expected_vi_6010106 = vi_alt_comp(nth_percentiles_6010106)
expected_vi_942030011 = vi_alt_comp(nth_percentiles_942030011)
expected_vi_intermittent = vi_alt_comp(nth_percentiles_intermittent)
expected_vi_synthetic = vi_alt_comp(synthetic_nth_percentile_flows)
n = [2,5,10,15,20,25,30,33,35,40,45,50,55,60,65,66,70,75,80,85,90,95,99]
test_compute_variability_index(nth_percentiles_2849991, n, expected_vi_2849991)
test_compute_variability_index(nth_percentiles_6010106, n, expected_vi_6010106)
test_compute_variability_index(nth_percentiles_942030011, n, expected_vi_942030011)
test_compute_variability_index(nth_percentiles_intermittent, n, expected_vi_intermittent)
test_compute_variability_index(synthetic_nth_percentile_flows, n, expected_vi_synthetic)


Computed Variability Index: 0.3240433471759944, Expected: 0.3240433471759944
 -> Test passed (OK)
Computed Variability Index: 0.16655062113682473, Expected: 0.16655062113682473
 -> Test passed (OK)
Computed Variability Index: 0.2907085052560591, Expected: 0.2907085052560591
 -> Test passed (OK)
Computed Variability Index: 0.3242400317023827, Expected: 0.3242400317023827
 -> Test passed (OK)
Computed Variability Index: 1.7476551248374688, Expected: 1.7476551248374688
 -> Test passed (OK)


### Test computing slope of flow duration curve

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_slope_fdc

def test_compute_slope_fdc(nth_percentiles, n, expected):
    slope_fdc = compute_slope_fdc(nth_percentiles, n)
    print(f"Computed Slope of FDC: {slope_fdc}, Expected: {expected}")
    assert np.isclose(slope_fdc, expected, atol=0.05*expected), f"Expected Slope of FDC {expected} but got {slope_fdc}"
    print(" -> Test passed (OK)")

test_compute_slope_fdc(nth_percentiles_2849991, n, 0.02420339)
test_compute_slope_fdc(nth_percentiles_6010106, n, 0.01092122)
test_compute_slope_fdc(nth_percentiles_942030011, n, 0.01870782)

daily_retro_data_synthetic = daily_retro_data_intermittent.copy()
daily_retro_data_synthetic.iloc[0:5500, 0] = 0 # creating a case where 33th percentile is zero
test_compute_slope_fdc(compute_percentiles(daily_retro_data_synthetic['streamflow'], n), n, 0.36366939)


Computed Slope of FDC: 0.02403006739756811, Expected: 0.02420339
 -> Test passed (OK)
Computed Slope of FDC: 0.010813743966680917, Expected: 0.01092122
 -> Test passed (OK)
Computed Slope of FDC: 0.018498671654567, Expected: 0.01870782
 -> Test passed (OK)
Computed Slope of FDC: 0.37365356041423736, Expected: 0.36366939
 -> Test passed (OK)


### Test computing Flashiness Index

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_flashiness_index

def test_compute_flashiness_index(data, expected):
    flashiness_index = compute_flashiness_index(data)
    print(f"Computed Flashiness Index: {flashiness_index}, \nExpected RBI: {expected}")
    assert np.allclose(flashiness_index, expected), f"Expected {expected}, but got {flashiness_index}"
    print(" -> Test passed (OK)")

test_compute_flashiness_index(daily_retro_data_2849991, [0.06771682, 0.06489289, 0.06685558, 0.05710084, 0.4184071])
test_compute_flashiness_index(daily_retro_data_6010106, [0.03040827, 0.03345583, 0.03539209, 0.03394035, 0.2653095])
test_compute_flashiness_index(daily_retro_data_942030011, [0.2066203, 0.2153028, 0.2568637, 0.2322091, 0.1633343])

Computed Flashiness Index: [0.06771681866189519, 0.06489289123300265, 0.06685558490787699, 0.057100836613097325, 0.4184071038381231], 
Expected RBI: [0.06771682, 0.06489289, 0.06685558, 0.05710084, 0.4184071]
 -> Test passed (OK)
Computed Flashiness Index: [0.0304082689713012, 0.03345583078471513, 0.035392092733244424, 0.03394035044421549, 0.2653094507932088], 
Expected RBI: [0.03040827, 0.03345583, 0.03539209, 0.03394035, 0.2653095]
 -> Test passed (OK)
Computed Flashiness Index: [0.2066203432332358, 0.21530279662674318, 0.25686373050766154, 0.23220913442812424, 0.16333433053950466], 
Expected RBI: [0.2066203, 0.2153028, 0.2568637, 0.2322091, 0.1633343]
 -> Test passed (OK)


### Test computing half flow date

In [ ]:
from dataflow_nwm_retro_indices_transformation import get_half_flow_date

def test_get_half_flow_date(data, expected):
    half_flow_date = get_half_flow_date(data)
    print(f"Computed Half Flow Date: {half_flow_date}")
    assert half_flow_date == expected, f"Expected Half Flow Date {expected} but got {half_flow_date}"
    print(" -> Test passed (OK)")

test_get_half_flow_date(daily_retro_data_2849991[daily_retro_data_2849991['water_year'] == 1980], pd.to_datetime('1980-03-06'))
test_get_half_flow_date(daily_retro_data_2849991[daily_retro_data_2849991['water_year'] == 1981], pd.to_datetime('1981-03-30'))
test_get_half_flow_date(daily_retro_data_2849991[daily_retro_data_2849991['water_year'] == 1982], pd.to_datetime('1982-03-08'))
test_get_half_flow_date(daily_retro_data_6010106[daily_retro_data_6010106['water_year'] == 2015], pd.to_datetime('2015-05-23'))
test_get_half_flow_date(daily_retro_data_942030011[daily_retro_data_942030011['water_year'] == 1982], pd.to_datetime('1981-12-23'))
test_get_half_flow_date(daily_retro_data_942030011[daily_retro_data_942030011['water_year'] == 1987], pd.to_datetime('1987-04-19'))


Computed Half Flow Date: 1980-03-06 00:00:00
 -> Test passed (OK)
Computed Half Flow Date: 1981-03-30 00:00:00
 -> Test passed (OK)
Computed Half Flow Date: 1982-03-08 00:00:00
 -> Test passed (OK)
Computed Half Flow Date: 2015-05-23 00:00:00
 -> Test passed (OK)
Computed Half Flow Date: 1981-12-23 00:00:00
 -> Test passed (OK)
Computed Half Flow Date: 1987-04-19 00:00:00
 -> Test passed (OK)


In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_half_flow_date

def test_get_half_flow_date(data, manual_mean, manual_cov):
    mean_half_flow_date, cov_half_flow_date = compute_half_flow_date(data)
    print(f"Computed Mean Half Flow Date: {mean_half_flow_date}, Expected: {manual_mean}")
    print(f"Computed CoV Half Flow Date: {cov_half_flow_date}, Expected: {manual_cov}")
    assert np.allclose(mean_half_flow_date, manual_mean, atol=0.01*manual_mean), f"Expected Mean Half Flow Date {manual_mean} but got {mean_half_flow_date}"
    assert np.allclose(cov_half_flow_date, manual_cov, atol=0.05*manual_cov), f"Expected CoV Half Flow Date {manual_cov} but got {cov_half_flow_date}"
    print(" -> Test passed (OK)")

test_get_half_flow_date(daily_retro_data_2849991, 80.44, 23.05)
test_get_half_flow_date(daily_retro_data_6010106, 113.98, 18.82)
test_get_half_flow_date(daily_retro_data_942030011, 97.65, 41.32)

Computed Mean Half Flow Date: 80.44186046511628, Expected: 80.44
Computed CoV Half Flow Date: 22.77803193396204, Expected: 23.05
 -> Test passed (OK)
Computed Mean Half Flow Date: 113.97674418604652, Expected: 113.98
Computed CoV Half Flow Date: 18.604643489856038, Expected: 18.82
 -> Test passed (OK)
Computed Mean Half Flow Date: 97.65116279069767, Expected: 97.65
Computed CoV Half Flow Date: 40.83266256972809, Expected: 41.32
 -> Test passed (OK)


### Test computing the start of flood season

In [717]:
np.random.seed(43)
dates = pd.date_range("2000-01-01", "2005-12-31", freq="D")
n = len(dates)
base = np.random.normal(loc=10, scale=2, size=n)

values = base.copy()
for year in [2000, 2001, 2002, 2003, 2004, 2005]:
    start = pd.Timestamp(f"{year}-11-01") # assuming flood season starts on Nov 1st for all years
    end = start + pd.Timedelta(days=179) 
    mask = (dates >= start) & (dates <= end)
    values[mask] += 20

synthetic_timeseries_with_fixed_fss = pd.Series(values, index=dates, name="streamflow").to_frame()
synthetic_ts_with_fixed_fss_and_nans = synthetic_timeseries_with_fixed_fss.copy()
synthetic_ts_with_fixed_fss_and_nans.loc["2001-01-01":"2001-01-15"] = np.nan
daily_retro_data_synthetic_2 = daily_retro_data_2849991.copy()
daily_retro_data_synthetic_2.loc["1991-05-01":"1991-06-05"] = np.nan # more than 30 days of nans in a year

In [ ]:
from dataflow_nwm_retro_indices_transformation import get_median_starting_date_of_flood_season

def test_get_median_starting_date_of_flood_season(daily_retro_data, expected_date):
    median_date = get_median_starting_date_of_flood_season(daily_retro_data)
    print(f"Computed Median Starting Date of Flood Season: {median_date}, Expected: {expected_date}")
    assert median_date == expected_date, f"Expected Median Starting Date of Flood Season {expected_date} but got {median_date}"
    print(" -> Test passed (OK)")
test_get_median_starting_date_of_flood_season(synthetic_timeseries_with_fixed_fss, 305)
test_get_median_starting_date_of_flood_season(synthetic_ts_with_fixed_fss_and_nans, 305)
test_get_median_starting_date_of_flood_season(daily_retro_data_2849991, 330)
test_get_median_starting_date_of_flood_season(daily_retro_data_synthetic_2, 176)
print("Is a year with more than 30 days of NaNs included:", get_median_starting_date_of_flood_season(daily_retro_data_2849991.loc["1991"]))
test_get_median_starting_date_of_flood_season(daily_retro_data_6010106, 100)
test_get_median_starting_date_of_flood_season(daily_retro_data_942030011, 131)
test_get_median_starting_date_of_flood_season(daily_retro_data_intermittent, 330)

Computed Median Starting Date of Flood Season: 305, Expected: 305
 -> Test passed (OK)
Computed Median Starting Date of Flood Season: 305, Expected: 305
 -> Test passed (OK)
Computed Median Starting Date of Flood Season: 330, Expected: 330
 -> Test passed (OK)
Computed Median Starting Date of Flood Season: 176, Expected: 176
 -> Test passed (OK)
Is a year with more than 30 days of NaNs included: nan
Computed Median Starting Date of Flood Season: 100, Expected: 100
 -> Test passed (OK)
Computed Median Starting Date of Flood Season: 131, Expected: 131
 -> Test passed (OK)
Computed Median Starting Date of Flood Season: 330, Expected: 330
 -> Test passed (OK)


### Test computing annual zero flow days count

In [ ]:
from dataflow_nwm_retro_indices_transformation import count_zero_flow_days

def test_count_zero_flow_days(data, expected):
    zero_flow_days = count_zero_flow_days(data)
    print(f"Computed Zero Flow Days: {zero_flow_days}, Expected: {expected}")
    assert np.isclose(zero_flow_days, expected, atol=0.01*expected), f"Expected Zero Flow Days {expected} but got {zero_flow_days}"
    print(" -> Test passed (OK)")
    
test_count_zero_flow_days(daily_retro_data_2849991, 0)
test_count_zero_flow_days(daily_retro_data_6010106, 0)
test_count_zero_flow_days(daily_retro_data_942030011, 0)
test_count_zero_flow_days(daily_retro_data_intermittent, 0.3255814)



Computed Zero Flow Days: 0.0, Expected: 0
 -> Test passed (OK)
Computed Zero Flow Days: 0.0, Expected: 0
 -> Test passed (OK)
Computed Zero Flow Days: 0.0, Expected: 0
 -> Test passed (OK)
Computed Zero Flow Days: 0.32558139534883723, Expected: 0.3255814
 -> Test passed (OK)


### Test computing annual high flow days count and average duration of high flow events

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_high_flow_count_and_duration

def test_compute_high_flow_count_and_duration(data, threshold, expected_count, expected_duration):
    high_flow_days_n, duration_high_flow_event = compute_high_flow_count_and_duration(data, threshold)
    print(f"Computed High Flow Days Count/Year: {high_flow_days_n}, Expected: {expected_count}")
    print(f"Computed Duration of High Flow Event: {duration_high_flow_event}, Expected: {expected_duration}")
    assert np.allclose(high_flow_days_n, expected_count, atol=1e-2), f"Expected {expected_count}, but got {high_flow_days_n}"
    assert np.allclose(duration_high_flow_event, expected_duration, atol=1e-2), f"Expected {expected_duration}, but got {duration_high_flow_event}"
    print(" -> Test passed (OK)")

test_compute_high_flow_count_and_duration(daily_retro_data_2849991, 9*daily_retro_data_2849991['streamflow'].median(), expected_count=4.767366, expected_duration=4.880952)
test_compute_high_flow_count_and_duration(daily_retro_data_6010106, 9*daily_retro_data_6010106['streamflow'].median(), expected_count=0.02325544, expected_duration=1)
test_compute_high_flow_count_and_duration(daily_retro_data_942030011, 9*daily_retro_data_942030011['streamflow'].median(), expected_count=6.8371, expected_duration=3.585366)

Total high flow events in all years: 205
Computed High Flow Days Count/Year: 4.767441860465116, Expected: 4.767366
Computed Duration of High Flow Event: 4.880952380952381, Expected: 4.880952
 -> Test passed (OK)
Total high flow events in all years: 1
Computed High Flow Days Count/Year: 0.023255813953488372, Expected: 0.02325544
Computed Duration of High Flow Event: 1.0, Expected: 1
 -> Test passed (OK)
Total high flow events in all years: 294
Computed High Flow Days Count/Year: 6.837209302325581, Expected: 6.8371
Computed Duration of High Flow Event: 3.5853658536585367, Expected: 3.585366
 -> Test passed (OK)


### Test computing annual low flow days count and average duration of low flow events

In [ ]:
from dataflow_nwm_retro_indices_transformation import compute_low_flow_count_and_duration

def test_compute_low_flow_count_and_duration(data, threshold, expected_count, expected_duration):
    low_flow_days_n, duration_low_flow_event = compute_low_flow_count_and_duration(data, threshold)
    print(f"Computed Low Flow Days Count/Year: {low_flow_days_n}, Expected: {expected_count}")
    print(f"Computed Duration of Low Flow Event: {duration_low_flow_event}, Expected: {expected_duration}")
    assert np.allclose(low_flow_days_n, expected_count, atol=1e-2), f"Expected {expected_count}, but got {low_flow_days_n}"
    assert np.allclose(duration_low_flow_event, expected_duration, atol=1e-2), f"Expected {expected_duration}, but got {duration_low_flow_event}"
    print(" -> Test passed (OK)")
    
test_compute_low_flow_count_and_duration(daily_retro_data_2849991, 0.2*daily_retro_data_2849991['streamflow'].mean(), expected_count=7.651041, expected_duration=21.93333)
test_compute_low_flow_count_and_duration(daily_retro_data_6010106, 0.2*daily_retro_data_6010106['streamflow'].mean(), expected_count=0, expected_duration=0)
test_compute_low_flow_count_and_duration(daily_retro_data_942030011, 0.2*daily_retro_data_942030011['streamflow'].mean(), expected_count=1.209283, expected_duration=10.4)


Total low flow events in all years: 329
Computed Low Flow Days Count/Year: 7.651162790697675, Expected: 7.651041
Computed Duration of Low Flow Event: 21.933333333333334, Expected: 21.93333
 -> Test passed (OK)
Total low flow events in all years: 0
Computed Low Flow Days Count/Year: 0.0, Expected: 0
Computed Duration of Low Flow Event: 0, Expected: 0
 -> Test passed (OK)
Total low flow events in all years: 52
Computed Low Flow Days Count/Year: 1.2093023255813953, Expected: 1.209283
Computed Duration of Low Flow Event: 10.4, Expected: 10.4
 -> Test passed (OK)
